In [1]:
# load_dotenv reads API keys/secrets from a local .env file into environment variables
from dotenv import load_dotenv
# OpenAI client used to call the chat completions API
from openai import OpenAI
# Used to render model responses as formatted Markdown in the notebook output
from IPython.display import Markdown, display

In [2]:
# Load environment variables from .env; override=True lets .env values replace any existing shell env vars
load_dotenv(override=True)

True

In [3]:
import os
# Fetch the OpenAI API key that load_dotenv() placed into the environment
openai_api_key = os.getenv('OPENAI_API_KEY')

# Sanity check: confirm the key loaded correctly without printing the whole secret
if openai_api_key:
    print(f"OpenAI API KEY exists and begins {openai_api_key[:8]}")
else:
    print("OPENAI API KEY not set- please head to the troubleshooting guide in the set up folder")

OpenAI API KEY exists and begins sk-proj-


In [4]:
# Create the OpenAI client; it automatically picks up OPENAI_API_KEY from the environment
openai = OpenAI()

In [5]:
# Chat completions API expects a list of role/content messages; this is a single user turn
messages = [{"role": "user", "content": "Tell me a fun fact"}]

In [6]:
# Inspect the messages list before sending it to the API
messages

[{'role': 'user', 'content': 'Tell me a fun fact'}]

In [7]:
# Send the messages to the model and print just the reply text
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

Fun fact: Octopuses have three hearts—two pump blood to the gills, and one pumps it to the rest of the body. When an octopus swims, its main heart may slow down or even pause briefly.


In [8]:
# Step 1 of a self-quiz pattern: ask the model to generate a hard question (answer not included)
question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question"
messages = [{"role": "user", "content": question}]

In [9]:
# Call the model to actually generate the question text, overwriting `question` with the model's output
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
question = response.choices[0].message.content
print(question)

In a sealed room there are three switches on the outside and three matching lamps inside; you may enter the room only once. Switch on exactly one switch for 10 minutes, then turn it off. Without changing the state of the other two switches (one remains off and the other remains off), enter the room and observe the states of the lamps. After your single observation, determine which switch controls which lamp—show the logic that leads to a unique mapping.


In [10]:
# Step 2: wrap the generated question in a fresh message list so it can be asked to a model
messages = [{"role":"user", "content":question}]
messages

[{'role': 'user',
  'content': 'In a sealed room there are three switches on the outside and three matching lamps inside; you may enter the room only once. Switch on exactly one switch for 10 minutes, then turn it off. Without changing the state of the other two switches (one remains off and the other remains off), enter the room and observe the states of the lamps. After your single observation, determine which switch controls which lamp—show the logic that leads to a unique mapping.'}]

In [11]:
# Answer the question using a stronger model, and keep the answer for later evaluation
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
answer = response.choices[0].message.content
print(answer)

Use **heat** as the extra clue.

### Procedure
1. Turn **switch 1** on for **10 minutes**.
2. Turn **switch 1** off.
3. Leave **switch 2** and **switch 3** off the whole time.
4. Enter the room once and inspect the three lamps.

### What you can observe
For each lamp, there are three possible states:

- **Lit** → controlled by the switch that is still on.  
- **Unlit but warm** → controlled by the switch that was on for 10 minutes, then turned off.  
- **Unlit and cold** → controlled by one of the two switches that were never turned on.

### Logical deduction
Suppose inside you find:

- one lamp **on**
- one lamp **off but warm**
- one lamp **off and cold**

Then the mapping is uniquely determined:

- the **on lamp** = the switch you left on,
- the **warm lamp** = the switch you turned on for 10 minutes and then off,
- the **cold lamp** = the remaining switch.

### Why this is unique
Because:
- only one switch was left on, so only its lamp can be lit;
- only one switch was ever energiz

In [12]:
# Render the answer as formatted Markdown instead of raw text
display(Markdown(answer))

Use **heat** as the extra clue.

### Procedure
1. Turn **switch 1** on for **10 minutes**.
2. Turn **switch 1** off.
3. Leave **switch 2** and **switch 3** off the whole time.
4. Enter the room once and inspect the three lamps.

### What you can observe
For each lamp, there are three possible states:

- **Lit** → controlled by the switch that is still on.  
- **Unlit but warm** → controlled by the switch that was on for 10 minutes, then turned off.  
- **Unlit and cold** → controlled by one of the two switches that were never turned on.

### Logical deduction
Suppose inside you find:

- one lamp **on**
- one lamp **off but warm**
- one lamp **off and cold**

Then the mapping is uniquely determined:

- the **on lamp** = the switch you left on,
- the **warm lamp** = the switch you turned on for 10 minutes and then off,
- the **cold lamp** = the remaining switch.

### Why this is unique
Because:
- only one switch was left on, so only its lamp can be lit;
- only one switch was ever energized long enough to heat a lamp, so only its lamp can be warm;
- the remaining switch was never on, so its lamp must be cold.

Thus, with one visit and one timed on/off action, you get a one-to-one assignment of all three switches to all three lamps.

In [13]:
# Step 3: build a grading prompt that hands the question + candidate answer to a judge model
message = f"""
here is a question:

{question}

And here is a possible answer that might be correct or incorrect:
{answer}

please evaluate if the answer is correct or incorrect.
"""

print(message)


here is a question:

In a sealed room there are three switches on the outside and three matching lamps inside; you may enter the room only once. Switch on exactly one switch for 10 minutes, then turn it off. Without changing the state of the other two switches (one remains off and the other remains off), enter the room and observe the states of the lamps. After your single observation, determine which switch controls which lamp—show the logic that leads to a unique mapping.

And here is a possible answer that might be correct or incorrect:
Use **heat** as the extra clue.

### Procedure
1. Turn **switch 1** on for **10 minutes**.
2. Turn **switch 1** off.
3. Leave **switch 2** and **switch 3** off the whole time.
4. Enter the room once and inspect the three lamps.

### What you can observe
For each lamp, there are three possible states:

- **Lit** → controlled by the switch that is still on.  
- **Unlit but warm** → controlled by the switch that was on for 10 minutes, then turned off. 

In [14]:
# Use the most capable model as a "judge" to fact-check the earlier answer (LLM-as-judge pattern)
messages = [{
    "role":"user",
    "content":message
}]
response = openai.chat.completions.create(model="gpt-5.4", messages=messages)
print(response.choices[0].message.content)

The proposed answer is **incorrect** for the question as stated.

### Why it’s incorrect
The answer’s logic depends on having **three distinct lamp states** available when you enter:

1. **on**
2. **off but warm**
3. **off and cold**

But in its own procedure, it says:

- turn **switch 1** on for 10 minutes,
- then turn **switch 1** off,
- leave **switch 2** and **switch 3** off the whole time,
- then enter.

That means **all three switches are off** when you enter the room.

So inside the room, the best you can possibly observe is:

- one lamp **off but warm**
- two lamps **off and cold**

There is **no lamp lit**, because no switch was left on.

### Consequence
You can identify only:

- the lamp connected to **switch 1**: it’s the **warm** one.

But the other two lamps are both **cold and off**, so you **cannot distinguish** which belongs to switch 2 and which belongs to switch 3.

So the mapping is **not unique**.

### What a correct solution would do
A correct strategy is:

1. Turn

In [15]:
# First create the message
messages = [{
    "role":"user",
    "content":"Help me build a business Using agentic ai. I'm not sure what type of business I want. Please provide 5 industries to focus on."
}]
messages

[{'role': 'user',
  'content': "Help me build a business Using agentic ai. I'm not sure what type of business I want. Please provide 5 industries to focus on."}]

In [16]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
answer = response.choices[0].message.content
print(answer)

Sure—here are 5 promising industries where **agentic AI** (AI agents that can plan, take actions, and complete workflows across tools) can create clear value. I’ll include *why each fits* and *what the AI agents would typically do*.

## 1) Healthcare Admin & Patient Support (Non-clinical)
**Why it fits:** Heavy paperwork, scheduling complexity, and lots of repetitive communication.  
**Agentic AI could do:**
- Schedule/reschedule appointments, triage and route requests (non-medical)
- Insurance eligibility/billing support workflows (document gathering + status updates)
- Manage patient messages, forms, referrals, and follow-ups

## 2) Legal Operations (Document + Process Automation)
**Why it fits:** Contract review, intake, e-sign workflows, filings, and case updates are process-heavy.  
**Agentic AI could do:**
- Client intake (collect docs, categorize matters, draft summaries)
- Contract clause extraction + redlines suggestions
- Filing packet assembly, deadline tracking, and status 

In [17]:
business_area = f"""
I have a background in finance. How does that apply to one of the options.
{answer}
"""
messages = [{"role": "user", "content": business_area}]
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
answer2 = response.choices[0].message.content
print(answer2)

Given your finance background, the best “hook” to translate that into agentic AI is usually **what you already understand well: workflows with inputs/outputs, risk/compliance, reconciliation, exceptions, and decisioning under constraints**. Of the 5 options you listed, the ones that map most naturally to finance skills are typically **(5) Logistics/Supply Chain** and **(2) Legal Operations**, with a close second depending on your finance niche.

### How finance applies to each option (and which one fits best)
#### 5) Logistics & Supply Chain Coordination (often the strongest fit)
**Why finance maps directly**
- Lots of **financial-grade processes**: invoices, bills of lading, proof-of-delivery, credits/chargebacks, disputes.
- High **exception rates** (delays, misroutes, damaged goods) → very similar to handling **reconciliations and anomaly resolution** in finance.
- Clear **metrics** (on-time %, cost per shipment, cycle time, SLA compliance).

**How agentic AI would “work” in this do